In [0]:
from datetime import timedelta
from delta.tables import DeltaTable
from pyspark.sql import functions as F, types as T

# set and refresh target and designate hospital billing table
TARGET = "opsanalytics_adb_workspace01.capacity_modeling.service_groups"
CHARGE_SOURCE = "datahub_prod_ext_oci_dashbd_user.dashbd_user.oe_charge_detail"
spark.catalog.refreshTable(TARGET)
# get current date
run_date = spark.range(1).select(F.current_date().alias("run_date")).first().run_date

In [0]:
# read the current service group mappings
current = spark.table(TARGET)

# oracle secrets for jdbc and OAO_PRODUCTION password
oracle_jdbc_url = dbutils.secrets.get(scope="oao_secrets", key="ORACLE_JDBC_URL")
oracle_password = dbutils.secrets.get(scope="oao_secrets", key="OAO_PRODUCTION")

# JDBC table reading function
def oracle_table(dbtable):
    return (
        spark.read.format("jdbc")
        .option("url", oracle_jdbc_url)
        .option("dbtable", dbtable)
        .option("user", "OAO_PRODUCTION")
        .option("password", oracle_password)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("oracle.net.ssl_server_dn_match", "true")
        .option("fetchsize", "10000")
        .load()
    )

# format strings for comparisons
def normalized(column):
    value = F.trim(F.col(column).cast("string"))
    return F.when(F.length(value) == 0, F.lit(None)).otherwise(value)

# read in clarity dep ref rows
clarity_rows = (
    oracle_table("(SELECT DEPARTMENT_ID, LOC_NAME, EXTERNAL_NAME, SERVICE_GROUP "
                 "FROM DASHBD_USER.CLARITY_DEP_REF) dept_ref")
    .select(
        normalized("DEPARTMENT_ID").alias("EPIC_DEPT_ID"),
        normalized("LOC_NAME").alias("LOC_NAME"),
        normalized("EXTERNAL_NAME").alias("EXTERNAL_NAME"),
        normalized("SERVICE_GROUP").alias("SERVICE_GROUP"),
    )
    .collect()
)

# define clarity schemna
clarity_schema = T.StructType([
    T.StructField("EPIC_DEPT_ID", T.StringType(), True),
    T.StructField("LOC_NAME", T.StringType(), True),
    T.StructField("EXTERNAL_NAME", T.StringType(), True),
    T.StructField("SERVICE_GROUP", T.StringType(), True),
])

# create clarity dep ref dataframe
clarity_dep_ref = spark.createDataFrame(clarity_rows, clarity_schema)

# check for missing data or duplicate EPIC IDs
if clarity_dep_ref.count() == 0:
    raise ValueError("Clarity department reference is empty")
if clarity_dep_ref.groupBy("EPIC_DEPT_ID").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Clarity department reference has duplicate department IDs")

# create active mapping df as mappings with null valid to values and exclude ED
active = (
    current.filter(F.col("VALID_TO").isNull())
    .withColumn("EPIC_DEPT_ID", normalized("EPIC_DEPT_ID"))
)
tracked = active.filter(
    ~F.upper(F.coalesce(F.col("EXTERNAL_NAME"), F.lit(""))).contains("EMERGENCY")
)

# check that all tracked mappings have exactly 1 epic id mapping
if tracked.filter(F.col("EPIC_DEPT_ID").isNull()).limit(1).count():
    raise ValueError("Active service group mapping has a blank department ID")
if tracked.groupBy("EPIC_DEPT_ID").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Active service group mapping has duplicate department IDs")

In [0]:
# give current and clarity dep ref values distinct names so the comparison is explicit
old = tracked.select(
    "EPIC_DEPT_ID",
    normalized("LOC_NAME").alias("old_loc"),
    normalized("EXTERNAL_NAME").alias("old_name"),
    normalized("SERVICE_GROUP").alias("old_group"),
    F.col("VALID_FROM").cast("date").alias("old_valid_from"),
)

ref = clarity_dep_ref.select(
    "EPIC_DEPT_ID",
    F.col("LOC_NAME").alias("new_loc"),
    F.col("EXTERNAL_NAME").alias("new_name"),
    F.col("SERVICE_GROUP").alias("new_group"),
)

# Inner join comparison = IDs present in both active mappings and Oracle
changes = (
    old.join(ref, "EPIC_DEPT_ID", "inner")
    .filter(
        ~F.col("old_loc").eqNullSafe(F.col("new_loc"))
        | ~F.col("old_name").eqNullSafe(F.col("new_name"))
        | ~F.col("old_group").eqNullSafe(F.col("new_group"))
    )

)

# check for missing valid from
if changes.filter(F.col("old_valid_from").isNull()).limit(1).count():
    raise ValueError("A changed mapping has a missing or future VALID_FROM date")

# show the proposed old/new values before the write cell. 
print(f"Changed mappings: {changes.count()}")
display(changes.select("EPIC_DEPT_ID", "old_loc", "new_loc", "old_name", "new_name", "old_group", "new_group"))

In [0]:
# Left anti join = reference IDs with no active target mapping. 
new_candidates = clarity_dep_ref.join(active.select("EPIC_DEPT_ID"), "EPIC_DEPT_ID", "left_anti")

# Show all missing active IDs first whether there is a bed charge or not
print(f"New department candidates: {new_candidates.count()}")
display(new_candidates)

# get list of candidate EPIC IDs
candidate_ids = [row.EPIC_DEPT_ID for row in new_candidates.select("EPIC_DEPT_ID").collect()]

# set schema for EPIC ID bed charge check
bed_date_schema = T.StructType([
    T.StructField("EPIC_DEPT_ID", T.StringType(), False),
    T.StructField("first_bed_service_date", T.DateType(), True),
])

# check oe_charge_detail for bed charges in candidate EPIC ID deps
if candidate_ids:
    charge = spark.table(CHARGE_SOURCE)
    charge_aggregates = (
        charge.select("EPIC_DEPT_ID", "BILLING_CAT_DESC", "SERVICE_DATE")
        .filter(F.col("EPIC_DEPT_ID").isin(candidate_ids))
        .filter(F.col("BILLING_CAT_DESC") == "BED CHARGES")
        .filter(F.col("SERVICE_DATE") >= "20240601")
        .groupBy("EPIC_DEPT_ID")
        .agg(
            F.min("SERVICE_DATE").alias("FIRST_SERVICE_DATE"),
        )
        .select(
            normalized("EPIC_DEPT_ID").alias("EPIC_DEPT_ID"),
            F.to_date("FIRST_SERVICE_DATE", "yyyyMMdd").alias("first_bed_service_date")
        )
        .collect()
    )
    bed_date_rows = [
        (row.EPIC_DEPT_ID, row.first_bed_service_date) for row in charge_aggregates
    ]
else:
    bed_date_rows = []
bed_dates = spark.createDataFrame(bed_date_rows, bed_date_schema)

# display new units with bedcharges
new_with_bed_charges = new_candidates.join(F.broadcast(bed_dates), "EPIC_DEPT_ID", "inner")
print(f"New departments with bed charges: {new_with_bed_charges.count()}")
display(new_with_bed_charges)

In [0]:
# validate the rows that would be written
changed_rows = changes.select(
    "EPIC_DEPT_ID",
    F.col("new_loc").alias("LOC_NAME"),
    F.col("new_name").alias("EXTERNAL_NAME"),
    F.col("new_group").alias("SERVICE_GROUP"),
    "old_valid_from",
    F.lit(run_date).alias("valid_from"),
)
new_rows = new_with_bed_charges.select(
    "EPIC_DEPT_ID", "LOC_NAME", "EXTERNAL_NAME", "SERVICE_GROUP",
    F.lit(None).cast("date").alias("old_valid_from"),
    F.col("first_bed_service_date").alias("valid_from"),
)
# check for missing location external name or service group before adding
to_write = changed_rows.unionByName(new_rows)
if to_write.filter(
    F.col("LOC_NAME").isNull()
    | F.col("EXTERNAL_NAME").isNull()
    | F.col("SERVICE_GROUP").isNull()
).limit(1).count():
    raise ValueError("A mapping to write is missing a location, department name, or service group")

# get older changes and run date changes
older_changes = changed_rows.filter(F.col("old_valid_from") < F.lit(run_date))
run_date_changes = changed_rows.filter(F.col("old_valid_from") == F.lit(run_date))

# stage function to prep merging changes and new rows
def stage(frame, operation, merge_key):
    return frame.select(
        F.lit(operation).alias("operation"),
        (F.col("EPIC_DEPT_ID") if merge_key else F.lit(None).cast("string")).alias("merge_key"),
        "EPIC_DEPT_ID", "LOC_NAME", "EXTERNAL_NAME", "SERVICE_GROUP", "valid_from",
    )

staged = (
    stage(older_changes, "close", True)
    .unionByName(stage(run_date_changes, "replace", True))
    .unionByName(stage(older_changes, "insert", False))
    .unionByName(stage(new_rows, "insert", False))
)

# check for updates and merge if they exist
if to_write.limit(1).count():
    target = DeltaTable.forName(spark, TARGET)
    (
        target.alias("t")
        .merge(
            staged.alias("s"),
            "CAST(t.EPIC_DEPT_ID AS STRING) = s.merge_key AND t.VALID_TO IS NULL",
        )
        .whenMatchedUpdate(
            condition="s.operation = 'close'",
            set={"VALID_TO": F.lit(run_date - timedelta(days=1))},
        )
        .whenMatchedUpdate(
            condition="s.operation = 'replace'",
            set={
                "LOC_NAME": F.col("s.LOC_NAME"),
                "EXTERNAL_NAME": F.col("s.EXTERNAL_NAME"),
                "SERVICE_GROUP": F.col("s.SERVICE_GROUP"),
            },
        )
        .whenNotMatchedInsert(
            condition="s.operation = 'insert'",
            values={
                "LOC_NAME": F.col("s.LOC_NAME"),
                "EPIC_DEPT_ID": F.col("s.EPIC_DEPT_ID"),
                "EXTERNAL_NAME": F.col("s.EXTERNAL_NAME"),
                "SERVICE_GROUP": F.col("s.SERVICE_GROUP"),
                "VALID_FROM": F.col("s.valid_from"),
                "VALID_TO": F.lit(None).cast(current.schema["VALID_TO"].dataType),
            },
        )
        .execute()
    )
    print("Service group mappings updated")
else:
    print("No service group mappings to update")
